In [ ]:
using Plots
using LinearAlgebra
using BenchmarkTools
using Revise
using May28Project

# Laplacian Construction

## Row by Row Construction

In [ ]:
Lx = 1;
Ly = 2;
nx = 5; # (nx-1)×(ny-1) interior points
ny = 10;
Nx = nx-1;
Ny = ny-1;

x = LinRange(0, Lx, nx + 1)
y = LinRange(0, Ly, ny + 1)

@show Δx = x[2] - x[1];
@show Δy = y[2] - y[1];

L1 = assemble_laplacian2d_row(Δx, Δy, Nx, Ny);
display(L1)

In [ ]:
Matrix(L1)

## Kroenecker Construction

In [ ]:
Lx = 1;
Ly = 2;
nx = 5; # (nx-1)×(ny-1) interior points
ny = 10;
Nx = nx-1;
Ny = ny-1;

x = LinRange(0, Lx, nx + 1)
y = LinRange(0, Ly, ny + 1)

@show Δx = x[2] - x[1];
@show Δy = y[2] - y[1];

L2 = assemble_laplacian2d_kron(Δx, Δy, Nx, Ny);
display(L2)

In [ ]:
@show norm(L1 - L2)

# Solving the Poisson Equation

In [ ]:
nx = 25;
ny = 50;
Nx = nx-1;
Ny = ny-1;

Lx = 1;
Ly = 1;

x = LinRange(0, Lx, nx + 1)
y = LinRange(0, Ly, ny + 1)

Δx = x[2] - x[1];
Δy = y[2] - y[1];

xy = [[x_,y_] for x_ in x, y_ in y]
xy_interior = [[x_,y_] for x_ in x[2:end-1], y_ in y[2:end-1]]

L = assemble_laplacian2d_kron(Δx, Δy, Nx, Ny);
A = -L;

# construct exact solution and f
uex = X-> X[1]*(1-X[1])*X[2]*(1-X[2])
f = X-> 2*X[2]*(1-X[2])+2* X[1]*(1-X[1]);

Lattice is stored as a 2D array of points in 2D:

In [ ]:
xy_interior

In [ ]:
f.(xy_interior)

Store this as a column vector:

In [ ]:
# both work:
# B = f.(xy_interior)[:]
B = vec(f.(xy_interior))

In [ ]:
U = A\B

Undo the column stacking:

In [ ]:
u = reshape(U, Nx, Ny)

Notice, we plotted with the adjoint here:

In [ ]:
contourf(x[2:end-1], y[2:end-1], u')
xlabel!("x")
ylabel!("y")
xlims!(0, 1)
ylims!(0, 1)
title!("Numerical Solution")

In [ ]:
contourf(x, y, uex.(xy)')
xlabel!("x")
ylabel!("y")
xlims!(0, 1)
ylims!(0, 1)
title!("Exact Solution")

In [ ]:
@show norm(u - uex.(xy_interior))

# Performance 

In [ ]:
n_vals = [10, 20, 40, 80, 160, 320, 640, 1280];
# errors = zeros(length(n_vals))
for (i, n) in enumerate(n_vals)
    nx = n;
    ny = n;
    Nx = nx-1;
    Ny = ny-1;
    Lx = 1;
    Ly = 1;
    x = LinRange(0, Lx, nx + 1)
    y = LinRange(0, Ly, ny + 1)
    Δx = x[2] - x[1];
    Δy = y[2] - y[1];
    xy = [[x_,y_] for x_ in x, y_ in y]
    xy_interior = [[x_,y_] for x_ in x[2:end-1], y_ in y[2:end-1]]
    L = assemble_laplacian2d_kron(Δx, Δy, Nx, Ny);
    A = -L;
    uex = X-> X[1]*(1-X[1])*X[2]*(1-X[2])
    f = X-> 2*X[2]*(1-X[2])+2* X[1]*(1-X[1]);
    B = vec(f.(xy_interior))
    @show n;

    @btime U = A\B;
    # u = reshape(U, Nx, Ny)
    # errors[i] = norm(u - uex.(xy_interior))
end

Looks a little slower than $\mathrm{O}(N)$ where $N = n^2$ in computational time.

If we use a dense matrix, we see $\mathrm{O}(N^3)$ scaling:

In [ ]:
n_vals = [5, 10, 20, 40, 80];
# errors = zeros(length(n_vals))
for (i, n) in enumerate(n_vals)
    nx = n;
    ny = n;
    Nx = nx-1;
    Ny = ny-1;
    Lx = 1;
    Ly = 1;
    x = LinRange(0, Lx, nx + 1)
    y = LinRange(0, Ly, ny + 1)
    Δx = x[2] - x[1];
    Δy = y[2] - y[1];
    xy = [[x_,y_] for x_ in x, y_ in y]
    xy_interior = [[x_,y_] for x_ in x[2:end-1], y_ in y[2:end-1]]
    L = assemble_laplacian2d_kron(Δx, Δy, Nx, Ny);
    A = Matrix(-L);
    uex = X-> X[1]*(1-X[1])*X[2]*(1-X[2])
    f = X-> 2*X[2]*(1-X[2])+2* X[1]*(1-X[1]);
    B = vec(f.(xy_interior))
    @show n;

    @btime U = A\B;
    # u = reshape(U, Nx, Ny)
    # errors[i] = norm(u - uex.(xy_interior))
end